In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
df_total = pd.read_csv("data/nodes_total_48.csv")
df_compute = pd.read_csv("data/nodes_compute_48.csv")

### computation time vs number of nodes for each experiment

In [ ]:
df = df_compute.copy()

node_columns = [col for col in df.columns if col != 'experiment']
node_counts = [int(col.replace('*', '')) for col in node_columns]

# Create the plot
plt.figure(figsize=(12, 8))

# Plot each experiment as a separate line
colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown', 'pink', 'gray', 'cyan', 'magenta']
markers = ['o', 's', '^', 'D', 'x', 'v', '<', '>', 'P', '*']

for i, exp_num in enumerate(df['experiment']):
    # Get computation times for this experiment
    times = [df.iloc[i][col] for col in node_columns]
    
    plt.plot(node_counts, times, 
            #  color=colors[i], 
            #  marker=markers[i], 
             linewidth=2, 
             markersize=8,
             label=f'Experiment {exp_num}')



# Customize the plot
plt.xlabel('Number of Nodes', fontsize=12, fontweight='bold')
plt.ylabel('Computation Time (seconds)', fontsize=12, fontweight='bold')
plt.title('Performance Scaling: Computation Time vs Number of Nodes', fontsize=14, fontweight='bold')

# Customize grid
plt.grid(True, alpha=0.3, linestyle='--')

# Set x-axis ticks to show actual node counts
plt.xticks(node_counts, [str(c) for c in node_counts])

# Add legend
plt.legend(loc='upper right', frameon=True, fancybox=True, shadow=True)

# Improve layout
plt.tight_layout()

# Optional: Save the plot
plt.savefig('figures/Nodes/48/nodes_computation_all.png', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()


# Print some basic statistics
print("\nPerformance Summary:")
print("=" * 50)
for i, exp_num in enumerate(df['experiment']):
    single_node_time = df.iloc[i]['1']
    max_nodes_time = df.iloc[i]['8']
    speedup = single_node_time / max_nodes_time
    print(f"Experiment {exp_num}:")
    print(f"  Single node: {single_node_time:.2f}s")
    print(f"  8 nodes: {max_nodes_time:.2f}s")
    print(f"  Speedup: {speedup:.2f}x")
    print()

In [ ]:
df = df_compute.copy()

# Extract node columns and calculate means
node_columns = [col for col in df.columns if col != 'experiment']
node_counts = [int(col.replace('*', '')) for col in node_columns]

# Calculate mean times for each node count
mean_times = []
std_times = []  # Also calculate standard deviation for error bars

for col in node_columns:
    mean_time = df[col].mean()
    std_time = df[col].std()
    mean_times.append(mean_time)
    std_times.append(std_time)
    print(f"Nodes {col.replace('*', '')}: Mean = {mean_time:.2f}s, Std = {std_time:.2f}s")

# Create the aggregated plot
plt.figure(figsize=(12, 8))

# Plot the mean line with error bars
plt.errorbar(node_counts, mean_times, yerr=std_times,
             color='darkblue', 
             marker='o', 
             linewidth=3, 
             markersize=10,
             capsize=8,
             capthick=2,
             elinewidth=2,
             label='Mean ± Std Dev')

# Fill area between mean ± std for visual appeal
plt.fill_between(node_counts, 
                 [m - s for m, s in zip(mean_times, std_times)],
                 [m + s for m, s in zip(mean_times, std_times)],
                 alpha=0.2, 
                 color='lightblue',
                 label='±1 Std Dev')

# Customize the plot
plt.xlabel('Number of Nodes', fontsize=12, fontweight='bold')
plt.ylabel('Mean Computation Time (seconds)', fontsize=12, fontweight='bold')
plt.title('Aggregated Performance Scaling: Mean Computation Time vs Number of Nodes', 
          fontsize=14, fontweight='bold')


# Customize grid
plt.grid(True, alpha=0.3, linestyle='--')

# Set x-axis ticks to show actual node counts
plt.xticks(node_counts, [str(c) for c in node_counts])

# Add legend
plt.legend(loc='upper right', frameon=True, fancybox=True, shadow=True)

# Add annotations for some key points
min_time_idx = mean_times.index(min(mean_times))

# Improve layout
plt.tight_layout()

# Optional: Save the plot
plt.savefig('figures/Nodes/48/nodes_computation_agg.png', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()


# Print detailed statistics
print("\n" + "="*60)
print("AGGREGATED PERFORMANCE ANALYSIS")
print("="*60)

# Calculate overall speedup
single_node_mean = mean_times[0]
best_mean = min(mean_times)
best_nodes = node_counts[mean_times.index(best_mean)]

print(f"Single node mean time: {single_node_mean:.2f}s")
print(f"Best performance: {best_mean:.2f}s at {best_nodes} nodes")
print(f"Overall speedup: {single_node_mean/best_mean:.2f}x")

print(f"\nDetailed breakdown:")
for i, nodes in enumerate(node_counts):
    speedup = single_node_mean / mean_times[i]
    efficiency = speedup / nodes * 100
    print(f"{nodes:2d} nodes: {mean_times[i]:7.2f}s (±{std_times[i]:6.2f}) | "
          f"Speedup: {speedup:5.2f}x | Efficiency: {efficiency:5.1f}%")

# Calculate ideal vs actual scaling
print(f"\nScaling Analysis:")
print(f"{'Nodes':<6} {'Ideal Time':<12} {'Actual Mean':<12} {'Scaling Eff':<12}")
print("-" * 48)
for i, nodes in enumerate(node_counts):
    ideal_time = single_node_mean / nodes
    actual_time = mean_times[i]
    scaling_eff = (ideal_time / actual_time) * 100
    print(f"{nodes:<6} {ideal_time:<12.2f} {actual_time:<12.2f} {scaling_eff:<12.1f}%")

In [ ]:
df = df_compute.copy()

# Extract node columns and prepare data for box plots
node_columns = [col for col in df.columns if col != 'experiment']
node_counts = [int(col.replace('*', '')) for col in node_columns]

# Prepare data for box plot - each column becomes a list of values
box_data = []
for col in node_columns:
    box_data.append(df[col].tolist())

# Create the box plot
plt.figure(figsize=(14, 10))

# Create box plots
box_plot = plt.boxplot(box_data, 
                       positions=range(1, len(node_columns) + 1),
                       showmeans=True,      # Show mean markers
                       meanline=False,      # Show means as points, not lines
                       widths=0.6)


# Customize other elements
for element in ['whiskers', 'fliers', 'caps']:
    plt.setp(box_plot[element], color='darkblue', linewidth=1.5)

# Customize means
plt.setp(box_plot['means'], marker='D', markerfacecolor='red', 
         markeredgecolor='darkred', markersize=8)

# Set x-axis labels to node counts
plt.xticks(range(1, len(node_columns) + 1), 
           [str(count) for count in node_counts])


# Customize the plot
plt.xlabel('Number of Nodes', fontsize=12, fontweight='bold')
plt.ylabel('Computation Time (seconds)', fontsize=12, fontweight='bold')
plt.title('Performance Distribution: Box and Whisker Plot\nAcross Different Node Counts', 
          fontsize=14, fontweight='bold')

# Add grid
plt.grid(True, alpha=0.3, linestyle='--', axis='y')

# Add a subtle background color
plt.gca().set_facecolor('#f8f9fa')

# Create custom legend explaining the box plot elements
legend_elements = [
    plt.Line2D([0], [0], color='orange', linewidth=2, label='Median'),
    plt.Line2D([0], [0], marker='D', color='red', markerfacecolor='red', 
               markersize=8, linestyle='None', label='Mean'),
    plt.Rectangle((0, 0), 1, 1, facecolor='white', alpha=0.7, 
                  edgecolor='black', label='IQR (25th-75th percentile)'),
    plt.Line2D([0], [0], color='darkblue', linewidth=1, label='Whiskers (1.5×IQR)')
]

plt.legend(handles=legend_elements, loc='upper right', 
           frameon=True, fancybox=True, shadow=True)

# Improve layout
plt.tight_layout()

# Optional: Save the plot
plt.savefig('figures/Nodes/48/nodes_computation_box_whiskers.png', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()

----

### Total Times:

In [ ]:
df = df_total.copy()

node_columns = [col for col in df.columns if col != 'experiment']
node_counts = [int(col.replace('*', '')) for col in node_columns]

# Create the plot
plt.figure(figsize=(12, 8))

# Plot each experiment as a separate line
colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown', 'pink', 'gray', 'cyan', 'magenta']
markers = ['o', 's', '^', 'D', 'x', 'v', '<', '>', 'P', '*']

for i, exp_num in enumerate(df['experiment']):
    # Get computation times for this experiment
    times = [df.iloc[i][col] for col in node_columns]
    
    plt.plot(node_counts, times, 
            #  color=colors[i], 
            #  marker=markers[i], 
             linewidth=2, 
             markersize=8,
             label=f'Experiment {exp_num}')



# Customize the plot
plt.xlabel('Number of Nodes', fontsize=12, fontweight='bold')
plt.ylabel('Total Time (seconds)', fontsize=12, fontweight='bold')
plt.title('Performance Scaling: Total Time vs Number of Nodes', fontsize=14, fontweight='bold')

# Set x-axis to log scale for better visualization of node progression

# Customize grid
plt.grid(True, alpha=0.3, linestyle='--')

# Set x-axis ticks to show actual node counts
plt.xticks(node_counts, [str(c) for c in node_counts])

# Add legend
plt.legend(loc='upper right', frameon=True, fancybox=True, shadow=True)

# Improve layout
plt.tight_layout()

# Optional: Save the plot
plt.savefig('figures/Nodes/48/nodes_total_all.png', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()


# Print some basic statistics
print("\nPerformance Summary:")
print("=" * 50)
for i, exp_num in enumerate(df['experiment']):
    single_node_time = df.iloc[i]['1']
    max_nodes_time = df.iloc[i]['8']
    speedup = single_node_time / max_nodes_time
    print(f"Experiment {exp_num}:")
    print(f"  Single node: {single_node_time:.2f}s")
    print(f"  8 nodes: {max_nodes_time:.2f}s")
    print(f"  Speedup: {speedup:.2f}x")
    print()

In [ ]:
df = df_total.copy()

# Extract node columns and calculate means
node_columns = [col for col in df.columns if col != 'experiment']
node_counts = [int(col.replace('*', '')) for col in node_columns]

# Calculate mean times for each node count
mean_times = []
std_times = []  # Also calculate standard deviation for error bars

for col in node_columns:
    mean_time = df[col].mean()
    std_time = df[col].std()
    mean_times.append(mean_time)
    std_times.append(std_time)
    print(f"Nodes {col.replace('*', '')}: Mean = {mean_time:.2f}s, Std = {std_time:.2f}s")

# Create the aggregated plot
plt.figure(figsize=(12, 8))

# Plot the mean line with error bars
plt.errorbar(node_counts, mean_times, yerr=std_times,
             color='darkblue', 
             marker='o', 
             linewidth=3, 
             markersize=10,
             capsize=8,
             capthick=2,
             elinewidth=2,
             label='Mean ± Std Dev')

# Fill area between mean ± std for visual appeal
plt.fill_between(node_counts, 
                 [m - s for m, s in zip(mean_times, std_times)],
                 [m + s for m, s in zip(mean_times, std_times)],
                 alpha=0.2, 
                 color='lightblue',
                 label='±1 Std Dev')

# Customize the plot
plt.xlabel('Number of Nodes', fontsize=12, fontweight='bold')
plt.ylabel('Mean Total Time (seconds)', fontsize=12, fontweight='bold')
plt.title('Aggregated Performance Scaling: Mean Total Time vs Number of Nodes', 
          fontsize=14, fontweight='bold')



# Set log scale for better visualization
# plt.xscale('log')

# Customize grid
plt.grid(True, alpha=0.3, linestyle='--')

# Set x-axis ticks to show actual node counts
plt.xticks(node_counts, [str(c) for c in node_counts])
# y_ticks = [200, 300, 400, 500, 600, 700, 800, 900, 1000, 1500, 2000, 3000, 4000]
# plt.yticks(y_ticks, [str(tick) for tick in y_ticks])

# Add legend
plt.legend(loc='upper right', frameon=True, fancybox=True, shadow=True)

# Add annotations for some key points
min_time_idx = mean_times.index(min(mean_times))

# Improve layout
plt.tight_layout()

# Optional: Save the plot
plt.savefig('figures/Nodes/48/nodes_total_agg.png', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()


# Print detailed statistics
print("\n" + "="*60)
print("AGGREGATED PERFORMANCE ANALYSIS")
print("="*60)

# Calculate overall speedup
single_node_mean = mean_times[0]
best_mean = min(mean_times)
best_nodes = node_counts[mean_times.index(best_mean)]

print(f"Single node mean time: {single_node_mean:.2f}s")
print(f"Best performance: {best_mean:.2f}s at {best_nodes} nodes")
print(f"Overall speedup: {single_node_mean/best_mean:.2f}x")

print(f"\nDetailed breakdown:")
for i, nodes in enumerate(node_counts):
    speedup = single_node_mean / mean_times[i]
    efficiency = speedup / nodes * 100
    print(f"{nodes:2d} nodes: {mean_times[i]:7.2f}s (±{std_times[i]:6.2f}) | "
          f"Speedup: {speedup:5.2f}x | Efficiency: {efficiency:5.1f}%")

# Calculate ideal vs actual scaling
print(f"\nScaling Analysis:")
print(f"{'Nodes':<6} {'Ideal Time':<12} {'Actual Mean':<12} {'Scaling Eff':<12}")
print("-" * 48)
for i, nodes in enumerate(node_counts):
    ideal_time = single_node_mean / nodes
    actual_time = mean_times[i]
    scaling_eff = (ideal_time / actual_time) * 100
    print(f"{nodes:<6} {ideal_time:<12.2f} {actual_time:<12.2f} {scaling_eff:<12.1f}%")

In [ ]:
df = df_total.copy()

# Extract node columns and prepare data for box plots
node_columns = [col for col in df.columns if col != 'experiment']
node_counts = [int(col.replace('*', '')) for col in node_columns]

# Prepare data for box plot - each column becomes a list of values
box_data = []
for col in node_columns:
    box_data.append(df[col].tolist())

# Create the box plot
plt.figure(figsize=(14, 10))

# Create box plots
box_plot = plt.boxplot(box_data, 
                       positions=range(1, len(node_columns) + 1),
                       showmeans=True,      # Show mean markers
                       meanline=False,      # Show means as points, not lines
                       widths=0.6)


# Customize other elements
for element in ['whiskers', 'fliers', 'caps']:
    plt.setp(box_plot[element], color='darkblue', linewidth=1.5)

# Customize means
plt.setp(box_plot['means'], marker='D', markerfacecolor='red', 
         markeredgecolor='darkred', markersize=8)

# Set x-axis labels to node counts
plt.xticks(range(1, len(node_columns) + 1), 
           [str(count) for count in node_counts])


# Customize the plot
plt.xlabel('Number of Nodes', fontsize=12, fontweight='bold')
plt.ylabel('Total Time (seconds)', fontsize=12, fontweight='bold')
plt.title('Performance Distribution: Box and Whisker Plot\nAcross Different Node Counts', 
          fontsize=14, fontweight='bold')

# Add grid
plt.grid(True, alpha=0.3, linestyle='--', axis='y')

# Add a subtle background color
plt.gca().set_facecolor('#f8f9fa')

# Create custom legend explaining the box plot elements
legend_elements = [
    plt.Line2D([0], [0], color='orange', linewidth=2, label='Median'),
    plt.Line2D([0], [0], marker='D', color='red', markerfacecolor='red', 
               markersize=8, linestyle='None', label='Mean'),
    plt.Rectangle((0, 0), 1, 1, facecolor='white', alpha=0.7, 
                  edgecolor='black', label='IQR (25th-75th percentile)'),
    plt.Line2D([0], [0], color='darkblue', linewidth=1, label='Whiskers (1.5×IQR)')
]

plt.legend(handles=legend_elements, loc='upper right', 
           frameon=True, fancybox=True, shadow=True)

# Improve layout
plt.tight_layout()

# Optional: Save the plot
plt.savefig('figures/Nodes/48/nodes_total_box_whiskers.png', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()